# 02 · Metric and baseline

**Project FORESIGHT — NorthBay Living**

Before any model. Section 7.1 of the brief is explicit about the order: fix how success is
measured, build the simple thing, and only then earn the right to something complex.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from foresight.config import get_settings
from foresight.eda import PALETTE, apply_house_style

warnings.filterwarnings("ignore", category=FutureWarning)
apply_house_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

settings = get_settings()
settings.processed_dir

In [ ]:
from foresight.baseline import naive_last_value_forecast, seasonal_naive_forecast
from foresight.features import build_base_features, build_supervised_frame, build_weekly_calendar
from foresight.metrics import evaluate_forecast, mape, wape

panel = pd.read_parquet(settings.processed_dir / "weekly_panel.parquet")
calendar = pd.read_parquet(settings.processed_dir / "calendar.parquet")

weekly_calendar = build_weekly_calendar(calendar)
base = build_base_features(panel)
horizons = tuple(range(1, settings.horizon_weeks + 1))
supervised = build_supervised_frame(base, weekly_calendar, horizons)

len(supervised)

## 1. Why WAPE and not MAPE

MAPE is the metric stakeholders recognise. It is also undefined on a week that sold nothing,
and enormous on a week that sold one unit — which describes a large part of this catalogue.

In [ ]:
sample = supervised.sample(20_000, random_state=0)
naive = sample["units_lag_0"].fillna(0.0)

mape_value, coverage = mape(sample["y"], naive)
print(f"WAPE  {wape(sample['y'], naive):.3f}   (defined on 100% of rows)")
print(f"MAPE  {mape_value:.3f}   (defined on only {coverage:.1%} of rows)")

low_volume = sample[sample["y"].between(1, 2)]
print(f"\nOn weeks selling 1-2 units, MAPE = {mape(low_volume['y'], low_volume['units_lag_0'].fillna(0))[0]:.2f}")

MAPE is dominated by the smallest SKUs and cannot be computed at all on a meaningful share
of rows. **WAPE is the metric the model is selected on**; MAPE is reported alongside it for
familiarity, always with its coverage.

## 2. The baseline: seasonal-naive

`forecast(t + h) = units(t + h - 52)` — the same week one year earlier. Because the horizon
is far shorter than a year, that value is always known at forecast time.

In [ ]:
result = seasonal_naive_forecast(supervised)
last_value = naive_last_value_forecast(supervised)

seasonal_metrics = evaluate_forecast(supervised["y"], result.predictions)
naive_metrics = evaluate_forecast(supervised["y"], last_value)

pd.DataFrame(
    {
        "seasonal-naive": seasonal_metrics.to_dict(),
        "last-value naive": naive_metrics.to_dict(),
    }
).loc[["wape", "mape", "mape_coverage", "bias_relative", "mae", "rmse"]]

## 3. The baseline's fatal flaw is not its accuracy — it is its bias

In [ ]:
print(f"seasonal-naive bias: {seasonal_metrics.bias_relative:+.1%}")
print(f"fallback rate (SKUs with < 52 weeks of history): {result.fallback_rate:.1%}")

A bias of roughly **−18%** means the baseline forecasts about 18% light *every single week*.
NorthBay's demand is growing year on year, so last year's week is structurally too low.

A forecast that is consistently light guarantees chronic under-ordering — which is exactly
the stockout problem the client described in their brief. Accuracy and bias have to be read
together: a model can post a respectable WAPE and still be unusable.

## 4. Where the baseline struggles most

In [ ]:
frame = supervised.copy()
frame["baseline"] = result.predictions

by_horizon = frame.groupby("horizon").apply(
    lambda group: wape(group["y"], group["baseline"]), include_groups=False
).rename("WAPE")

by_category = frame.groupby("category").apply(
    lambda group: wape(group["y"], group["baseline"]), include_groups=False
).rename("WAPE").sort_values()

display(by_horizon.to_frame())
display(by_category.to_frame())

The bar is now fixed and measured. Anything built next has to clear it honestly —
on a rolling-origin backtest, not on the training period.

---

Next: [`03_model.ipynb`](03_model.ipynb) — features, leakage, and beating the bar.